In [1]:
import pandas as pd
import re


# Creación de la base de datos

In [2]:
comprobantes = pd.read_excel('Comprobantes detallados 2025 SIIGO.xlsx', header=7)
comprobantes.head(10)

,Secuencia,Fecha elaboración,Código contable,Cuenta contable,Identificación,Sucursal,Nombre tercero,Descripción,Detalle,Centro de costo,Débito,Crédito
0,Comprobante: A-272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,220940.19,220940.19
1,1,08/01/2025,14350101.0,Mercancías no fabricadas,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,LINEA PLATINO INT SENCILLO CEB,Prod: 30103 Cant: 1.00,NaN,1557.98,0.00
2,2,08/01/2025,61350520.0,Ajuste de Inventarió,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,1557.98
3,3,08/01/2025,14350101.0,Mercancías no fabricadas,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,LINEA PLATINO TOMA CORRIENTE DOBLE CEB,Prod: 30107 Cant: 1.00,NaN,0.00,1929.78
4,4,08/01/2025,61350520.0,Ajuste de Inventarió,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,1929.78,0.00
5,5,08/01/2025,14350101.0,Mercancías no fabricadas,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,TOMA CORRIENTE DOBLE LINEA STAR BLACK CEB,Prod: NTST15T Cant: 10.00,NaN,0.00,70875.50
6,6,08/01/2025,61350520.0,Ajuste de Inventarió,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,70875.50,0.00
7,7,08/01/2025,14350101.0,Mercancías no fabricadas,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,INTERRUPTOR DOBLE NEGRO LINEA STAR BLACK CEB,Prod: LM5211 Cant: 10.00,NaN,93722.70,0.00
8,8,08/01/2025,61350520.0,Ajuste de Inventarió,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,93722.70
9,9,08/01/2025,14350101.0,Mercancías no fabricadas,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,CAJA DE INTEMPERIE 5800 3 SALIDAS 1/2 MAXWELL,Prod: 70189 Cant: 3.00,NaN,20528.70,0.00


In [3]:
# Extraer código de comprobante y propagarlo
comprobantes['Comprobante'] = comprobantes['Secuencia'].where(
    comprobantes['Secuencia'].astype(str).str.startswith('Comprobante:')
).ffill()
comprobantes['Comprobante'] = comprobantes['Comprobante'].str.replace('Comprobante: ', '', regex=False)

# Eliminar filas encabezado de comprobante
comprobantes = comprobantes[
    ~comprobantes['Secuencia'].astype(str).str.startswith('Comprobante:')
].reset_index(drop=True)

# Convertir columnas numéricas (Int64 soporta NaN)
comprobantes['Código contable'] = comprobantes['Código contable'].astype('Int64')
comprobantes['Identificación']  = comprobantes['Identificación'].astype('Int64')
comprobantes['Sucursal']        = comprobantes['Sucursal'].astype('Int64')

# Movimiento: crédito cuando débito=0, -débito cuando crédito=0
comprobantes['Movimiento'] = comprobantes.apply(
    lambda r: r['Crédito'] if r['Débito'] == 0 else -r['Débito'], axis=1
)

cols = ['Comprobante'] + [c for c in comprobantes.columns if c != 'Comprobante']
comprobantes = comprobantes[cols]
comprobantes.head(20)

,Comprobante,Secuencia,Fecha elaboración,Código contable,Cuenta contable,Identificación,Sucursal,Nombre tercero,Descripción,Detalle,Centro de costo,Débito,Crédito,Movimiento
0,A-272,1,08/01/2025,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,LINEA PLATINO INT SENCILLO CEB,Prod: 30103 Cant: 1.00,NaN,1557.98,0.00,-1557.98
1,A-272,2,08/01/2025,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,1557.98,1557.98
2,A-272,3,08/01/2025,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,LINEA PLATINO TOMA CORRIENTE DOBLE CEB,Prod: 30107 Cant: 1.00,NaN,0.00,1929.78,1929.78
3,A-272,4,08/01/2025,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,1929.78,0.00,-1929.78
4,A-272,5,08/01/2025,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,TOMA CORRIENTE DOBLE LINEA STAR BLACK CEB,Prod: NTST15T Cant: 10.00,NaN,0.00,70875.50,70875.50
5,A-272,6,08/01/2025,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,70875.50,0.00,-70875.50
6,A-272,7,08/01/2025,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,INTERRUPTOR DOBLE NEGRO LINEA STAR BLACK CEB,Prod: LM5211 Cant: 10.00,NaN,93722.70,0.00,-93722.70
7,A-272,8,08/01/2025,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,93722.70,93722.70
8,A-272,9,08/01/2025,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,CAJA DE INTEMPERIE 5800 3 SALIDAS 1/2 MAXWELL,Prod: 70189 Cant: 3.00,NaN,20528.70,0.00,-20528.70
9,A-272,10,08/01/2025,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,20528.70,20528.70


In [4]:
productos = pd.read_excel('Listado de productos _ Servicios .xlsx', header=6)

# Filtrar filas válidas
productos = productos[productos['Tipo'].isin(['Producto', 'Servicio'])].copy().reset_index(drop=True)

# Renombrar para consistencia
productos = productos.rename(columns={'Código': 'Código producto', 'Nombre': 'Nombre producto'})

print(f'Productos en catálogo: {len(productos)}')
productos.head(10)

Productos en catálogo: 2391


,Tipo,Código producto,Nombre producto,Referencia fábrica,Categoría,Estado,Impuesto cargo,Descripción larga,Es incluido,Inventariable,Saldo cantidades,Visible en facturas de venta,Stock mínimo,Precio de venta 1,Precio de venta 2
0,Producto,000001EM,CAJA MECANISMO 32MM ECO. EM,NaN,Productos,Activo,IVA 19%,NaN,Si,Si,0.0,SI,0.0,NaN,NaN
1,Producto,000527,"FACE PLATE SENCILLO PANDUIT M,GENERICA",NaN,Productos,Activo,IVA 19%,NaN,Si,Si,0.0,SI,0.0,NaN,NaN
2,Producto,00089,GUANTE CARNAZA SENCILLO CORTO M GENERICA,NaN,Productos,Activo,IVA 19%,NaN,Si,Si,0.0,SI,0.0,NaN,NaN
3,Producto,00091,GUANTE CARNAZA SENCILLO SOLDADOR M. GENERICA,NaN,Productos,Activo,IVA 19%,NaN,Si,Si,0.0,SI,0.0,NaN,NaN
4,Producto,006,LLANA METALICA M.GENERICA,NaN,Productos,Activo,IVA 19%,NaN,Si,Si,0.0,SI,0.0,NaN,NaN
5,Producto,0090044000013,CABLE AL 400MCM SERIE 8000 FACELEC,NaN,Productos,Activo,IVA 19%,NaN,Si,Si,0.0,SI,0.0,NaN,NaN
6,Producto,020030013,CABLE DUPLEX POLARIZADO 2X18 M. GEN,NaN,Productos,Activo,IVA 19%,NaN,Si,Si,0.0,SI,0.0,NaN,NaN
7,Producto,020060004,CABLE SOLDADOR #2 X MT PROCABLES,NaN,Productos,Activo,IVA 19%,NaN,Si,Si,0.0,SI,0.0,NaN,NaN
8,Producto,020060008,CABLE SOLDADOR 4AWG CU M GENE,NaN,Productos,Activo,IVA 19%,NaN,Si,Si,0.0,SI,0.0,NaN,NaN
9,Producto,020060011,CABLE SOLDADOR 6AWG M GEN,NaN,Productos,Activo,IVA 19%,NaN,Si,Si,0.0,SI,0.0,NaN,NaN


In [5]:
def norm_nombre_det(s):
    """Normaliza nombre extraído del campo Detalle del comprobante."""
    if pd.isna(s): return s
    s = str(s).strip()
    s = re.sub(r'\s+', ' ', s)                    # normalizar espacios múltiples
    s = re.sub(r'\s+Bod:.*$', '', s)               # quitar info de bodega
    s = re.sub(r'\s+M\.?\s*GEN(?:E(?:RICA)?)?$', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+M\.?\s*(CENTELSA|CONDUMEX|SYLVANIA|SULVANIA|MAXWELL|KLIC|MERCURY|LEGRAND|TUBOSA)$',
               '', s, flags=re.IGNORECASE)
    parts = s.split()
    if len(parts) > 1 and re.match(r'^[A-Z]{3,}$', parts[-1]):
        s = ' '.join(parts[:-1])
    return s.rstrip('. ')

def norm_nombre_cat(s):
    """Normaliza nombre del catálogo de productos."""
    if pd.isna(s): return s
    s = str(s).strip()
    s = re.sub(r'\s+M\.?\s*GEN(?:E(?:RICA)?)?$', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+M\.?\s+[A-Z]+$', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+(MAXWELL|CENTELSA|CONDUMEX|SYLVANIA|SULVANIA|TUBOSA|KLIC|MERCURY|LEGRAND|PHILLIPS)$',
               '', s, flags=re.IGNORECASE)
    return s.rstrip('. ')

productos['nombre_norm'] = productos['Nombre producto'].apply(norm_nombre_cat)

# Lookups
_prod_dedup  = productos.drop_duplicates('Código producto')
lookup_code  = _prod_dedup.set_index('Código producto')['Categoría']
lookup_exact = productos.drop_duplicates('Nombre producto').set_index('Nombre producto')['Código producto']
lookup_norm  = productos.drop_duplicates('nombre_norm').set_index('nombre_norm')['Código producto']

tiene_prod     = comprobantes['Detalle'].str.contains(r'Prod(?:ucto)?:', na=False)
total_con_prod = tiene_prod.sum()

# Extraer código y nombre desde Detalle
comprobantes['_codigo_det'] = comprobantes['Detalle'].str.extract(r'Prod:\s*(\S+)')
comprobantes['_nombre_det'] = comprobantes['Detalle'].str.extract(r'Producto:\s*(.+?)\s*Cant:', flags=re.IGNORECASE)
comprobantes['Código producto'] = pd.NA
comprobantes['Categoría']       = pd.NA

# --- Paso 1: Código directo (Prod: CODIGO) ---
mask1 = comprobantes['_codigo_det'].notna() & tiene_prod
comprobantes.loc[mask1, 'Código producto'] = comprobantes.loc[mask1, '_codigo_det']
comprobantes.loc[mask1, 'Categoría'] = comprobantes.loc[mask1, '_codigo_det'].map(lookup_code)

# --- Paso 2: Nombre exacto (Producto: NOMBRE) ---
sin2 = comprobantes['Código producto'].isna() & tiene_prod & comprobantes['_nombre_det'].notna()
m2   = comprobantes.loc[sin2, '_nombre_det'].map(lookup_exact)
idx2 = m2.index[m2.notna()]
comprobantes.loc[idx2, 'Código producto'] = m2[m2.notna()].values
comprobantes.loc[idx2, 'Categoría'] = comprobantes.loc[idx2, 'Código producto'].map(lookup_code)

# --- Paso 3: Nombre normalizado ---
sin3     = comprobantes['Código producto'].isna() & tiene_prod & comprobantes['_nombre_det'].notna()
det_norm = comprobantes.loc[sin3, '_nombre_det'].apply(norm_nombre_det)
m3       = det_norm.map(lookup_norm)
idx3     = m3.index[m3.notna()]
comprobantes.loc[idx3, 'Código producto'] = m3[m3.notna()].values
comprobantes.loc[idx3, 'Categoría'] = comprobantes.loc[idx3, 'Código producto'].map(lookup_code)

# Reporte
sin_final     = comprobantes['Código producto'].isna() & tiene_prod
total_matched = total_con_prod - sin_final.sum()
print(f'Paso 1 - Código directo:  {mask1.sum():5d} ({mask1.sum()/total_con_prod*100:.1f}%)')
print(f'Paso 2 - Nombre exacto:   {m2.notna().sum():5d} ({m2.notna().sum()/total_con_prod*100:.1f}%)')
print(f'Paso 3 - Nombre norm:     {m3.notna().sum():5d} ({m3.notna().sum()/total_con_prod*100:.1f}%)')
print(f'TOTAL matched:            {total_matched:5d} / {total_con_prod} ({total_matched/total_con_prod*100:.1f}%)')
print(f'DESCONOCIDO:              {sin_final.sum():5d} ({sin_final.sum()/total_con_prod*100:.1f}%)')

if sin_final.any():
    restantes = sorted(comprobantes.loc[sin_final, '_nombre_det'].dropna().unique())
    print(f'\nDescripciones sin match ({len(restantes)}):')
    for d in restantes: print(f'  - {d}')

# Asignar DESCONOCIDO y limpiar columnas auxiliares
comprobantes.loc[comprobantes['Código producto'].isna() & tiene_prod, 'Código producto'] = 'DESCONOCIDO'
comprobantes = comprobantes.drop(columns=['_codigo_det', '_nombre_det'])
comprobantes.head(10)

Paso 1 - Código directo:   8879 (27.7%)
Paso 2 - Nombre exacto:   20928 (65.3%)
Paso 3 - Nombre norm:      2061 (6.4%)
TOTAL matched:            31868 / 32036 (99.5%)
DESCONOCIDO:                168 (0.5%)

Descripciones sin match (18):
  - C. 1/0AWG FREETOX NEGRO X MTRS M. CENTELSA
  - C. 10AWG CU AZUL THHN-2 600V X MTRS M. CENTELSA
  - C. 10AWG CU BLANCO THHN-2 600V X MT.  CENTELSA
  - C. 10AWG VERDE FREETOX 600V  ROLLO X 100 CENTELSA
  - C. 2AWG FREETOX NEGRO X MTRS M. CENTELSA
  - C. 6AWG BLANCO FREETOX 600V X MT CENTELSA
  - C. TRENZADO 3*12AWG CU AZ/BL/VE 1000 THHN-2 CONDUMEX X MT
  - CABLE UTP CAT 6 AZUL M. LANPRO
  - CABLE VEHICULO 18 AWG CU AWM 105 AZUL CENTELSA
  - CINTA LED RGB 5MT CEB Bod: ALM03
  - CURVA PVC 3/4 PLEXIN Bod: ALM02
  - GRAPA PORTA TENSORA 1-1/4 M GENE
  - LED EMERGENCIA 10W DL 75 PCS M. SYLVANIA Bod: ALM03
  - LED HOGHBAY 150W DL GC015 DIM SYLVANIA
  - PANEL LED PASTEL INC 36W REDONDO 6500K CEB Bod: ALM02
  - PANEL LED PASTEL SP REDONDO 24W 6500K CEB Bod: AL

,Comprobante,Secuencia,Fecha elaboración,Código contable,Cuenta contable,Identificación,Sucursal,Nombre tercero,Descripción,Detalle,Centro de costo,Débito,Crédito,Movimiento,Código producto,Categoría
0,A-272,1,08/01/2025,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,LINEA PLATINO INT SENCILLO CEB,Prod: 30103 Cant: 1.00,NaN,1557.98,0.00,-1557.98,30103,Productos
1,A-272,2,08/01/2025,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,1557.98,1557.98,<NA>,<NA>
2,A-272,3,08/01/2025,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,LINEA PLATINO TOMA CORRIENTE DOBLE CEB,Prod: 30107 Cant: 1.00,NaN,0.00,1929.78,1929.78,30107,Productos
3,A-272,4,08/01/2025,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,1929.78,0.00,-1929.78,<NA>,<NA>
4,A-272,5,08/01/2025,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,TOMA CORRIENTE DOBLE LINEA STAR BLACK CEB,Prod: NTST15T Cant: 10.00,NaN,0.00,70875.50,70875.50,NTST15T,Productos
5,A-272,6,08/01/2025,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,70875.50,0.00,-70875.50,<NA>,<NA>
6,A-272,7,08/01/2025,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,INTERRUPTOR DOBLE NEGRO LINEA STAR BLACK CEB,Prod: LM5211 Cant: 10.00,NaN,93722.70,0.00,-93722.70,LM5211,Productos
7,A-272,8,08/01/2025,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,93722.70,93722.70,<NA>,<NA>
8,A-272,9,08/01/2025,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,CAJA DE INTEMPERIE 5800 3 SALIDAS 1/2 MAXWELL,Prod: 70189 Cant: 3.00,NaN,20528.70,0.00,-20528.70,70189,Productos
9,A-272,10,08/01/2025,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,20528.70,20528.70,<NA>,<NA>


# Procesamiento de datos

In [6]:
# Facturas de venta (FV) con cuenta de ingresos (41350101)
ventas = comprobantes[
    (comprobantes['Comprobante'].str.startswith('FV')) &
    (comprobantes['Código contable'] == 41350101)
].copy()

# Enriquecer con nombre del producto desde catálogo
ventas = ventas.merge(
    productos[['Código producto', 'Nombre producto']].drop_duplicates('Código producto'),
    on='Código producto', how='left'
)

# Columnas de fecha y tiempo
ventas['Fecha']      = pd.to_datetime(ventas['Fecha elaboración'], dayfirst=True)
ventas['Año']        = ventas['Fecha'].dt.year
ventas['Trimestre']  = ventas['Fecha'].dt.quarter
ventas['Mes']        = ventas['Fecha'].dt.month
ventas['Mes_nombre'] = ventas['Fecha'].dt.strftime('%b')
ventas['Semana']     = ventas['Fecha'].dt.isocalendar().week.astype(int)

# Cantidad vendida
ventas['Cantidad'] = ventas['Detalle'].str.extract(r'Cant:\s*([\d.]+)').astype(float)

print(f'Filas de ventas: {len(ventas)}')
print(f'Facturas únicas: {ventas["Comprobante"].nunique()}')
print(f'Clientes únicos: {ventas["Nombre tercero"].nunique()}')
print(f'Productos únicos: {ventas["Código producto"].nunique()}')
print(f'Venta total: ${ventas["Crédito"].sum():,.0f}')
ventas.head()

Filas de ventas: 9736
Facturas únicas: 3821
Clientes únicos: 1604
Productos únicos: 1047
Venta total: $975,835,445


,Comprobante,Secuencia,Fecha elaboración,Código contable,Cuenta contable,Identificación,Sucursal,Nombre tercero,Descripción,Detalle,...,Código producto,Categoría,Nombre producto,Fecha,Año,Trimestre,Mes,Mes_nombre,Semana,Cantidad
0,FV-2-8872,1,07/01/2025,41350101,Comercio al por mayor y al detal,79500411,0,WILLAN MUÑOZ,CAJA DE INTEMPERIE 5800 3 SALIDAS 1/2 MAXWELL,Producto: CAJA DE INTEMPERIE 5800 3 SALIDAS 1...,...,70189,Productos,CAJA DE INTEMPERIE 5800 3 SALIDAS 1/2 MAXWELL,2025-01-07,2025,1,1,Jan,2,4.0
1,FV-2-8872,2,07/01/2025,41350101,Comercio al por mayor y al detal,79500411,0,WILLAN MUÑOZ,TERMINAL METALICA (EMT) 1/2 ACERO MAXWELL,Producto: TERMINAL METALICA (EMT) 1/2 ACERO MA...,...,70143,Productos,TERMINAL METALICA (EMT) 1/2 ACERO MAXWELL,2025-01-07,2025,1,1,Jan,2,40.0
2,FV-2-8872,3,07/01/2025,41350101,Comercio al por mayor y al detal,79500411,0,WILLAN MUÑOZ,CAJA METALICA 5800 MAXWELL,Producto: CAJA METALICA 5800 MAXWELL Cant: 12.00,...,70181,Productos,CAJA METALICA 5800 MAXWELL,2025-01-07,2025,1,1,Jan,2,12.0
3,FV-2-8873,1,07/01/2025,41350101,Comercio al por mayor y al detal,900662485,0,TEINELEC SAS,SENSOR DE PARED DE 180°EN L 110-130V/AC KLIC,Producto: SENSOR DE PARED DE 180°EN L 110-130V...,...,SEN6,Productos,SENSOR DE PARED DE 180°EN L 110-130V/AC KLIC,2025-01-07,2025,1,1,Jan,2,2.0
4,FV-2-8873,2,07/01/2025,41350101,Comercio al por mayor y al detal,900662485,0,TEINELEC SAS,PANEL LED REDONDO 18W INCRUSTAR 6500K 165-240V...,Producto: PANEL LED REDONDO 18W INCRUSTAR 6500...,...,LN602-18W-220*10,Productos,PANEL LED REDONDO 18W INCRUSTAR 6500K 165-240V...,2025-01-07,2025,1,1,Jan,2,2.0


In [7]:
# --- comprobantes_enriquecidos.csv ---
# Todos los movimientos contables con producto y categoría
comprobantes_out = comprobantes[[
    'Comprobante', 'Fecha elaboración', 'Código contable', 'Cuenta contable',
    'Identificación', 'Nombre tercero', 'Descripción', 'Detalle',
    'Débito', 'Crédito', 'Movimiento',
    'Código producto', 'Categoría'
]]
comprobantes_out.to_csv('comprobantes_enriquecidos.csv', index=False)

# --- ventas.csv ---
# Solo facturas de venta, con columnas de fecha y producto para dashboards
ventas_out = ventas[[
    'Comprobante', 'Fecha', 'Año', 'Trimestre', 'Mes', 'Mes_nombre', 'Semana',
    'Identificación', 'Nombre tercero',
    'Código producto', 'Nombre producto', 'Categoría',
    'Descripción', 'Cantidad', 'Crédito'
]].rename(columns={'Crédito': 'Venta'})
ventas_out.to_csv('ventas.csv', index=False)

# --- productos.csv ---
productos.drop(columns=['nombre_norm'], errors='ignore').to_csv('productos.csv', index=False)

print(f'comprobantes_enriquecidos.csv  → {len(comprobantes_out):,} filas')
print(f'ventas.csv                     → {len(ventas_out):,} filas')
print(f'productos.csv                  → {len(productos):,} filas')
print()
print('Columnas ventas.csv:')
print(ventas_out.dtypes.to_string())

comprobantes_enriquecidos.csv  → 51,712 filas
ventas.csv                     → 9,736 filas
productos.csv                  → 2,391 filas

Columnas ventas.csv:
Comprobante                object
Fecha              datetime64[ns]
Año                         int32
Trimestre                   int32
Mes                         int32
Mes_nombre                 object
Semana                      int64
Identificación              Int64
Nombre tercero             object
Código producto            object
Nombre producto            object
Categoría                  object
Descripción                object
Cantidad                  float64
Venta                     float64
